# Stress, strain and Mohr's circle

*fluids · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/fluids/stress_strain)

Continuum mechanics of an elastic plate. Hooke's law is a rank-4 stiffness tensor $C_{ijkl}$ built from the Lamé constants with einsum; a loading ramp (tension, then shear) deforms the plate, the strain comes from the gradient of the displacement field, and the principal stresses from eigh — drawn as a stress ellipse and Mohr's circle.

**Vector form**

$$\boldsymbol\varepsilon = \tfrac12\big(\nabla\mathbf u + (\nabla\mathbf u)^\mathsf T\big),\qquad \boldsymbol\sigma = \mathsf C : \boldsymbol\varepsilon = \lambda\,(\operatorname{tr}\boldsymbol\varepsilon)\,\mathbf I + 2\mu\,\boldsymbol\varepsilon$$

**Index (tensor) form**

$$\varepsilon_{ij} = \tfrac12(\partial_i u_j + \partial_j u_i),\qquad \sigma_{ij} = C_{ijkl}\,\varepsilon_{kl},\qquad C_{ijkl} = \lambda\,\delta_{ij}\delta_{kl} + \mu\,(\delta_{ik}\delta_{jl} + \delta_{il}\delta_{jk}),\qquad \sigma_{ij}n_j = \sigma\,n_i$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $C_{ijkl}$ | `stiffness(lam, mu, delta)` | (3, 3, 3, 3) | three einsum outer products of $\delta$; 81 entries, only 2 independent constants |
| $S_{ijkl}$ | `compliance(C)` | (3, 3, 3, 3) | inverse of $C$ on symmetric tensors: $\varepsilon = S:\sigma$ (pinv of the 9×9 matrix) |
| $u_i = \varepsilon_{ij}x_j$ | `einsum('ij,...j->...i', eps, x)` | (n, n, 3, 3) | displacement of every material point |
| $\partial_i u_j$ | `gradient(u, x)` | (n, n, 3, 3, 3) | displacement gradient by finite differences (torch.gradient) |
| $\varepsilon_{ij}$ | `strain(grad_u)` | (..., 3, 3) | symmetric part of the displacement gradient |
| $C_{ijkl}\varepsilon_{kl}$ | `einsum('ijkl,...kl->...ij', C, eps)` | (..., 3, 3) | repeated $k, l$ ⇒ summed: Hooke's law |
| $\sigma_{ij}n_j = \sigma\,n_i$ | `torch.linalg.eigh(sigma)` | (3,), (3, 3) | principal stresses and their directions |

The plate is thin and its faces are free, so $\sigma_{zz} = 0$ (plane stress) — but it still gets thinner: $\varepsilon_{zz} = -\nu(\sigma_{xx} + \sigma_{yy})/E$. In the uniaxial half of the ramp you can read off Young's modulus $E = \sigma_{xx}/\varepsilon_{xx} = \mu(3\lambda + 2\mu)/(\lambda + \mu)$ and Poisson's ratio $\nu = -\varepsilon_{yy}/\varepsilon_{xx} = \lambda/2(\lambda + \mu)$. Adding shear rotates the principal axes away from $x$ and $y$; on Mohr's circle the points $(\sigma_{xx}, \tau_{xy})$ and $(\sigma_{yy}, -\tau_{xy})$ are the ends of a diameter, and the circle meets the axis at the principal stresses. The radius is the maximum in-plane shear stress, which governs yielding in ductile metals (Tresca). Displacements are tiny and are exaggerated in the drawing.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/fluids/stress_strain/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import lame, lame_stress, stiffness


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    E, nu = params["E"] * 1e9, params["poisson"]
    lam, mu = lame(E, nu)
    delta = tensors.kronecker(dev)  # (3, 3)
    C = stiffness(lam, mu, delta)  # (3, 3, 3, 3)
    x = tensors.block_points(9, dev)  # (9, 9, 3, 3)
    sigma_applied = tensors.loading(60, params["sigma_max"] * 1e6, params["tau_max"] * 1e6, dev)  # (T, 3, 3)

    result = solve.load_path(sigma_applied, C, x)

    eps, sigma = result["eps"][-1], result["sigma"][-1]
    print(f"λ = {lam / 1e9:.1f} GPa, μ = {mu / 1e9:.1f} GPa")
    print(f"C_ijkl symmetric: minor {bool((C - C.transpose(0, 1)).abs().max() < 1e-3)}, "
          f"major {bool((C - C.permute(2, 3, 0, 1)).abs().max() < 1e-3)}")  # fmt: skip
    mid = len(sigma_applied) // 2
    e_mid = result["eps"][mid]
    if e_mid[0, 0] != 0:
        print(f"uniaxial step: E = σ_xx/ε_xx = {sigma_applied[mid, 0, 0] / e_mid[0, 0] / 1e9:.1f} GPa, "
              f"ν = -ε_yy/ε_xx = {-e_mid[1, 1] / e_mid[0, 0]:.3f}")  # fmt: skip
    print(f"final strain (×1e-6): ε_xx {eps[0, 0] * 1e6:.0f}, ε_yy {eps[1, 1] * 1e6:.0f}, "
          f"ε_zz {eps[2, 2] * 1e6:.0f}, ε_xy {eps[0, 1] * 1e6:.0f}")  # fmt: skip
    print(
        "einsum C:ε vs λ tr(ε) δ + 2μ ε: max difference "
        f"{(sigma - lame_stress(eps, lam, mu, delta)).abs().max():.2e} Pa"
    )
    s1, s2 = result["in_plane"][-1].flip(0) / 1e6
    print(f"principal stresses: σ1 = {s1:.1f} MPa, σ2 = {s2:.1f} MPa, max shear {(s1 - s2) / 2:.1f} MPa")
    return plot.animate(x, result, out_dir / "stress_strain.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/fluids/stress_strain/equation.py
r"""Physics → tensor translation.

Linear elasticity of a solid. The displacement field u_i(x) gives the (small) strain tensor

    ε_{ij} = ½ (∂_i u_j + ∂_j u_i)

and Hooke's law links strain to stress through the rank-4 stiffness tensor (k, l summed):

    σ_{ij} = C_{ijkl} ε_{kl},      C_{ijkl} = λ δ_{ij} δ_{kl} + μ (δ_{ik} δ_{jl} + δ_{il} δ_{jk})

For an isotropic material this collapses to σ_{ij} = λ ε_{kk} δ_{ij} + 2 μ ε_{ij}.
The principal stresses are the eigenvalues of the symmetric matrix σ_{ij} (torch.linalg.eigh);
Mohr's circle draws the in-plane stress state as a circle of centre (σ_xx+σ_yy)/2.
"""

import torch


def lame(E: float, nu: float) -> tuple[float, float]:
    """Lamé constants from Young's modulus E and Poisson's ratio ν."""
    lam = E * nu / ((1 + nu) * (1 - 2 * nu))
    mu = E / (2 * (1 + nu))
    return lam, mu


def stiffness(lam: float, mu: float, delta: torch.Tensor) -> torch.Tensor:
    r"""C_{ijkl} = λ δ_{ij} δ_{kl} + μ (δ_{ik} δ_{jl} + δ_{il} δ_{jk}).   delta: (3, 3) → (3, 3, 3, 3)"""
    return (
        lam * torch.einsum("ij,kl->ijkl", delta, delta)
        + mu * torch.einsum("ik,jl->ijkl", delta, delta)
        + mu * torch.einsum("il,jk->ijkl", delta, delta)
    )


def strain(grad_u: torch.Tensor) -> torch.Tensor:
    r"""ε_{ij} = ½ (∂_i u_j + ∂_j u_i).   grad_u[..., i, j] = ∂_i u_j: (..., 3, 3) → (..., 3, 3)"""
    return 0.5 * (grad_u + grad_u.transpose(-1, -2))


def stress(C: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
    r"""σ_{ij} = C_{ijkl} ε_{kl}   (k, l summed).   (3,3,3,3), (..., 3, 3) → (..., 3, 3)"""
    return torch.einsum("ijkl,...kl->...ij", C, eps)


def lame_stress(eps: torch.Tensor, lam: float, mu: float, delta: torch.Tensor) -> torch.Tensor:
    r"""The same law written out: σ_{ij} = λ ε_{kk} δ_{ij} + 2 μ ε_{ij}."""
    trace = torch.einsum("...kk->...", eps)  # ε_kk, repeated k ⇒ summed
    return lam * torch.einsum("...,ij->...ij", trace, delta) + 2 * mu * eps


def principal_stresses(sigma: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Eigenvalues (ascending) and eigenvectors (columns) of the symmetric stress tensor."""
    return torch.linalg.eigh(sigma)


def mohr_circle(sigma: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    r"""In-plane Mohr circle: centre (σ_xx + σ_yy)/2 and radius \sqrt{((σ_xx - σ_yy)/2)^2 + σ_xy^2}."""
    sxx, syy, sxy = sigma[..., 0, 0], sigma[..., 1, 1], sigma[..., 0, 1]
    return 0.5 * (sxx + syy), torch.sqrt((0.5 * (sxx - syy)) ** 2 + sxy**2)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/fluids/stress_strain/tensors.py
"""Build the concrete tensors: Kronecker delta, the block's material points and the loading ramp."""

import torch


def kronecker(device: torch.device) -> torch.Tensor:
    """δ_ij: shape (3, 3)."""
    return torch.eye(3, device=device, dtype=torch.float64)


def block_points(n: int, device: torch.device) -> torch.Tensor:
    """Material points of a unit square plate (a few points thick): shape (n, n, 3, 3) → x_i at [ix, iy, iz, i].

    x and y run from -0.5 to 0.5 (in units of the plate size); z = -0.05, 0, 0.05 (a thin plate).
    """
    s = torch.linspace(-0.5, 0.5, n, device=device, dtype=torch.float64)
    z = torch.tensor([-0.05, 0.0, 0.05], device=device, dtype=torch.float64)
    xx, yy, zz = torch.meshgrid(s, s, z, indexing="ij")  # (n, n, 3) each
    return torch.stack([xx, yy, zz], dim=-1)


def loading(n_steps: int, sigma_max: float, tau_max: float, device: torch.device) -> torch.Tensor:
    """Applied stress σ_ij for each load step: shape (T, 3, 3).

    First half: uniaxial tension σ_xx ramps to σ_max. Second half: a shear σ_xy = σ_yx ramps to τ_max on top.
    σ_zz = 0 everywhere (plane stress: the faces of a thin plate are free).
    """
    s = torch.linspace(0.0, 1.0, n_steps, device=device, dtype=torch.float64)
    sigma = torch.zeros(n_steps, 3, 3, device=device, dtype=torch.float64)
    sigma[:, 0, 0] = sigma_max * torch.clamp(2 * s, max=1.0)
    sigma[:, 0, 1] = sigma[:, 1, 0] = tau_max * torch.clamp(2 * s - 1, min=0.0)
    return sigma


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/fluids/stress_strain/solve.py
"""Each load step: stress → strain → displacement field → strain from its gradient → stress → principal stresses."""

import torch

from .equation import principal_stresses, strain, stress


def compliance(C: torch.Tensor) -> torch.Tensor:
    """S_{ijkl} with ε = S : σ — the inverse of C on symmetric tensors (pseudo-inverse of C as a 9×9 matrix)."""
    return torch.linalg.pinv(C.reshape(9, 9)).reshape(3, 3, 3, 3)


def displacement(eps: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    r"""Uniform strain, no rotation: u_i = ε_{ij} x_j (j summed).   eps: (3, 3), x: (..., 3) → (..., 3)"""
    return torch.einsum("ij,...j->...i", eps, x)


def gradient(u: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """∂_i u_j by finite differences on the point grid: u, x (nx, ny, nz, 3) → (nx, ny, nz, 3, 3)."""
    spacing = [x[:, 0, 0, 0], x[0, :, 0, 1], x[0, 0, :, 2]]  # coordinates along each grid axis
    grads = torch.gradient(u, spacing=spacing, dim=(0, 1, 2))  # 3 tensors ∂_i u, each (nx, ny, nz, 3)
    return torch.stack(grads, dim=-2)  # [..., i, j] = ∂_i u_j


def load_path(sigma_applied: torch.Tensor, C: torch.Tensor, x: torch.Tensor) -> dict:
    """Run the whole loading ramp. Returns per-step tensors (T, ...)."""
    S = compliance(C)
    eps = torch.einsum("ijkl,tkl->tij", S, sigma_applied)  # (T, 3, 3) strain from Hooke's law inverted
    u = torch.stack([displacement(e, x) for e in eps])  # (T, nx, ny, nz, 3)
    eps_field = torch.stack([strain(gradient(ut, x)) for ut in u])  # (T, nx, ny, nz, 3, 3) from the displacement
    sigma = stress(C, eps_field.mean(dim=(1, 2, 3)))  # (T, 3, 3) back to stress: σ = C : ε
    values, vectors = principal_stresses(sigma)  # (T, 3), (T, 3, 3)
    in_plane, in_plane_dirs = principal_stresses(sigma[:, :2, :2])  # (T, 2), (T, 2, 2) the x-y block only
    return {"eps": eps, "u": u, "sigma": sigma, "principal": values, "directions": vectors,
            "in_plane": in_plane, "in_plane_dirs": in_plane_dirs}  # fmt: skip


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/fluids/stress_strain/plot.py
"""Three panels: deformed plate (exaggerated), stress ellipse with principal axes, and Mohr's circle."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Circle, Ellipse

from problems._common.plotting import frame_indices, save_gif


def animate(x: torch.Tensor, result: dict, path: Path) -> Path:
    mid = x.shape[2] // 2  # middle layer of the plate (z = 0)
    X = x[:, :, mid, :2].cpu().numpy()  # (n, n, 2)
    U = result["u"][:, :, :, mid, :2].cpu().numpy()  # (T, n, n, 2)
    sig = result["sigma"].cpu().numpy() / 1e6  # (T, 3, 3) in MPa
    vals = result["in_plane"].cpu().numpy() / 1e6  # (T, 2) in-plane principal stresses
    dirs = result["in_plane_dirs"].cpu().numpy()  # (T, 2, 2) columns = directions
    scale = 0.12 / max(np.abs(U).max(), 1e-12)  # exaggerate so the largest displacement is 12 % of the plate
    smax = max(np.abs(vals).max(), 1e-9)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(9, 3.3))
    for ax in (ax1, ax2):
        ax.set_aspect("equal")
        ax.grid(False)
        ax.set_xticks([])
        ax.set_yticks([])
    ax1.set_xlim(-0.75, 0.75)
    ax1.set_ylim(-0.75, 0.75)
    for k in range(X.shape[0]):
        ax1.plot(X[k, :, 0], X[k, :, 1], color="#d1d5db", lw=0.8)
        ax1.plot(X[:, k, 0], X[:, k, 1], color="#d1d5db", lw=0.8)
    rows = [ax1.plot([], [], color="#2563eb", lw=1)[0] for _ in range(X.shape[0])]
    cols = [ax1.plot([], [], color="#2563eb", lw=1)[0] for _ in range(X.shape[0])]
    title1 = ax1.set_title(" ", fontsize=9)

    ax2.set_xlim(-1.3, 1.3)
    ax2.set_ylim(-1.3, 1.3)
    ax2.set_title("stress ellipse, principal axes", fontsize=9)
    ellipse = ax2.add_patch(Ellipse((0, 0), 0, 0, fc="#93c5fd", ec="#1e40af", alpha=0.7))
    arrows = []

    centre, radius = sig[:, 0, 0] / 2 + sig[:, 1, 1] / 2, np.hypot((sig[:, 0, 0] - sig[:, 1, 1]) / 2, sig[:, 0, 1])
    lim = (centre + radius).max() * 1.15 + 1e-9
    ax3.set_xlim(-0.35 * lim, lim)
    ax3.set_ylim(-0.7 * lim, 0.7 * lim)
    ax3.set_aspect("equal")
    ax3.axhline(0, color="#9ca3af", lw=0.8)
    ax3.axvline(0, color="#9ca3af", lw=0.8)
    ax3.set_xlabel("σ (MPa)")
    ax3.set_ylabel("τ (MPa)")
    ax3.set_title("Mohr's circle", fontsize=9)
    circle = ax3.add_patch(Circle((0, 0), 0, fill=False, ec="#2563eb", lw=1.8))
    diameter = ax3.plot([], [], "o-", color="#dc2626", ms=4, lw=1)[0]
    ends = ax3.plot([], [], "s", color="#f59e0b", ms=5)[0]

    def update(step: int):
        D = X + scale * U[step]  # deformed positions (n, n, 2)
        for k in range(X.shape[0]):
            rows[k].set_data(D[k, :, 0], D[k, :, 1])
            cols[k].set_data(D[:, k, 0], D[:, k, 1])
        title1.set_text(f"σ_xx = {sig[step, 0, 0]:.0f}, σ_xy = {sig[step, 0, 1]:.0f} MPa  (×{scale:.0f})")

        v, d = vals[step] / smax, dirs[step]
        angle = np.degrees(np.arctan2(d[1, 1], d[0, 1]))  # direction of the larger principal stress
        ellipse.set_width(2 * max(abs(v[1]), 0.02))
        ellipse.set_height(2 * max(abs(v[0]), 0.02))
        ellipse.set_angle(angle)
        for a in arrows:
            a.remove()
        arrows.clear()
        for value, direction in zip(v, d.T):  # red arrows point out for tension, in for compression
            for sign in (1, -1):
                tip, tail = sign * direction * (abs(value) + 0.2), sign * direction * 0.2
                start, end = (tail, tip) if value > 0 else (tip, tail)
                if abs(value) > 0.02:
                    arrows.append(ax2.annotate("", end, start, arrowprops={"arrowstyle": "->", "color": "#dc2626"}))

        c, r = centre[step], radius[step]
        circle.set_center((c, 0))
        circle.set_radius(r)
        diameter.set_data([sig[step, 0, 0], sig[step, 1, 1]], [sig[step, 0, 1], -sig[step, 0, 1]])
        ends.set_data([c - r, c + r], [0, 0])  # principal stresses σ_2, σ_1

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(sig.shape[0], 60), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'E': 200.0,
    'poisson': 0.3,
    'sigma_max': 100.0,
    'tau_max': 60.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.fluids.stress_strain")]:
    del sys.modules[name]
main = importlib.import_module("problems.fluids.stress_strain.main")
problem_dir = Path("problems/fluids/stress_strain")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
